# Baseline Model Comparison

This notebook compares baseline same-game models for Tommy Award winner prediction using the top feature set from `Same_Game_And_Pregame_Logistic_Experiments.ipynb`.

The shared feature set is held fixed across models so the comparison focuses on model choice rather than feature choice.

Metric hierarchy used in this notebook:
- Primary: `game_level_accuracy`, `top3_accuracy`
- Secondary: `row_prauc`, `row_recall`, `row_precision`
- Probability quality: `row_brier`, `row_logloss`, `row_auc`
- Reference only: `row_accuracy`

In [22]:
# Import the libraries used for loading data, preprocessing,
# model training, and evaluation.
import importlib
import warnings
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore", message=".*penalty.*deprecated.*", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*penalty=l1.*", category=UserWarning)

# Basic configuration shared across the notebook.
INPUT_PATH = "Tommy_Award_Player_Game_Table.csv"
RANDOM_STATE = 42
MIN_TRAIN_SEASONS = 3
PRED_THRESHOLD = 0.5

In [24]:
# Load the dataset, define the shared same-game feature set,
# and build season-based train/test splits.
def load_dataset(path: str = INPUT_PATH) -> pd.DataFrame:
    df = pd.read_csv(path, dtype={"GAME_ID": str}).copy()
    df["game_date"] = pd.to_datetime(df["game_date"], format="mixed")

    # Keep only players who actually played in the game.
    df = df[df["minutes_decimal"] > 0].copy()

    # Recreate a few derived same-game features so this notebook can run
    # directly from the raw player-game table.
    mins = df["minutes_decimal"].clip(lower=1e-6)
    if "hustle_proxy" not in df.columns:
        df["hustle_proxy"] = df["reboundsOffensive"] + df["steals"] + df["blocks"]
    if "stocks_per_min" not in df.columns:
        df["stocks_per_min"] = df["stocks"] / mins
    if "stocks_rank" not in df.columns:
        df["stocks_rank"] = df.groupby("GAME_ID")["stocks"].rank(method="min", ascending=False)
    if "hustle_proxy_rank" not in df.columns:
        df["hustle_proxy_rank"] = df.groupby("GAME_ID")["hustle_proxy"].rank(method="min", ascending=False)

    if "season" not in df.columns:
        start_year = df["game_date"].dt.year.where(df["game_date"].dt.month >= 10, df["game_date"].dt.year - 1)
        df["season"] = start_year.astype(str) + "-" + (start_year + 1).astype(str).str[-2:]

    return df


def sorted_seasons(seasons: list[str]) -> list[str]:
    return sorted(seasons, key=lambda season: int(str(season).split("-")[0]))


def get_feature_columns(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    numeric_cols = [
        "minutes_decimal",
        "points",
        "reboundsOffensive",
        "reboundsDefensive",
        "reboundsTotal",
        "assists",
        "steals",
        "blocks",
        "turnovers",
        "foulsPersonal",
        "plusMinusPoints",
        "fieldGoalsMade",
        "fieldGoalsAttempted",
        "threePointersMade",
        "threePointersAttempted",
        "freeThrowsMade",
        "stocks",
        "points_per_min",
        "oreb_per_min",
        "reb_per_min",
        "ast_per_min",
        "stocks_per_min",
        "hustle_proxy",
        "points_rank",
        "reboundsOffensive_rank",
        "reboundsTotal_rank",
        "assists_rank",
        "steals_rank",
        "blocks_rank",
        "plusMinusPoints_rank",
        "minutes_decimal_rank",
        "stocks_rank",
        "hustle_proxy_rank",
    ]

    missing_cols = [col for col in numeric_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing feature columns: {missing_cols}")

    return numeric_cols, []


def walk_forward_season_splits(
    df: pd.DataFrame,
    min_train_seasons: int = MIN_TRAIN_SEASONS,
) -> list[tuple[list[str], str]]:
    seasons = sorted_seasons(df["season"].dropna().unique().tolist())
    splits = []
    for idx in range(min_train_seasons, len(seasons)):
        splits.append((seasons[:idx], seasons[idx]))
    return splits


def latest_season_holdout_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    seasons = sorted_seasons(df["season"].dropna().unique().tolist())
    train_df = df[df["season"].isin(seasons[:-1])].copy()
    test_df = df[df["season"] == seasons[-1]].copy()
    return train_df, test_df

In [25]:
# Build the preprocessing and baseline models.
# Linear models and SVM use scaling, while tree models do not need it.
def build_preprocessor(
    numeric_cols: list[str],
    categorical_cols: list[str],
    scale_numeric: bool,
) -> ColumnTransformer:
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_transformer = Pipeline(steps=numeric_steps)
    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
        ]
    )


def scaled_model_names() -> set[str]:
    return {"logistic", "ridge_logistic", "lasso_logistic", "svm"}


def available_model_names() -> tuple[list[str], dict[str, str]]:
    model_names = [
        "logistic",
        "ridge_logistic",
        "lasso_logistic",
        "random_forest",
        "svm",
    ]
    unavailable = {}

    for optional_model, module_name in [("lightgbm", "lightgbm"), ("xgboost", "xgboost")]:
        try:
            importlib.import_module(module_name)
            model_names.append(optional_model)
        except Exception as exc:
            unavailable[optional_model] = str(exc).strip()

    return model_names, unavailable


def build_model(
    model_name: str,
    numeric_cols: list[str],
    categorical_cols: list[str],
    scale_pos_weight: float,
) -> Pipeline:
    prep = build_preprocessor(
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        scale_numeric=model_name in scaled_model_names(),
    )

    if model_name == "logistic":
        clf = LogisticRegression(
            penalty=None,
            solver="lbfgs",
            max_iter=5000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )
    elif model_name == "ridge_logistic":
        clf = LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="lbfgs",
            max_iter=5000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )
    elif model_name == "lasso_logistic":
        clf = LogisticRegression(
            penalty="l1",
            C=1.0,
            solver="liblinear",
            max_iter=5000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )
    elif model_name == "random_forest":
        clf = RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    elif model_name == "svm":
        clf = SVC(
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )
    elif model_name == "lightgbm":
        LGBMClassifier = importlib.import_module("lightgbm").LGBMClassifier
        clf = LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            verbose=-1,
        )
    elif model_name == "xgboost":
        XGBClassifier = importlib.import_module("xgboost").XGBClassifier
        clf = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.80,
            colsample_bytree=0.80,
            random_state=RANDOM_STATE,
            n_jobs=4,
            scale_pos_weight=scale_pos_weight,
        )
    else:
        raise ValueError(f"Unsupported model_name: {model_name}")

    return Pipeline(
        steps=[
            ("prep", prep),
            ("clf", clf),
        ]
    )

In [26]:
# Score each model using both game-level ranking metrics and row-level classification metrics.
def score_predictions(
    scored_df: pd.DataFrame,
    threshold: float = PRED_THRESHOLD,
) -> dict[str, float]:
    pred_winners = (
        scored_df.sort_values(["GAME_ID", "pred_prob"], ascending=[True, False])
        .groupby("GAME_ID")
        .head(1)
        .copy()
    )
    top3 = (
        scored_df.sort_values(["GAME_ID", "pred_prob"], ascending=[True, False])
        .groupby("GAME_ID")
        .head(3)
        .copy()
    )

    y_true = scored_df["y"]
    y_prob = scored_df["pred_prob"].clip(1e-6, 1 - 1e-6)
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "game_level_accuracy": pred_winners["y"].mean(),
        "top3_accuracy": top3.groupby("GAME_ID")["y"].max().mean(),
        "row_prauc": average_precision_score(y_true, y_prob),
        "row_recall": recall_score(y_true, y_pred, zero_division=0),
        "row_precision": precision_score(y_true, y_pred, zero_division=0),
        "row_brier": brier_score_loss(y_true, y_prob),
        "row_logloss": log_loss(y_true, y_prob, labels=[0, 1]),
        "row_auc": roc_auc_score(y_true, y_prob),
        "row_accuracy": (y_pred == y_true).mean(),
    }


def fit_and_score_model(
    model_name: str,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    numeric_cols: list[str],
    categorical_cols: list[str],
) -> tuple[Pipeline, pd.DataFrame, dict[str, float]]:
    feature_cols = numeric_cols + categorical_cols
    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]
    y_train = train_df["y"]

    # Tree boosting uses this ratio to pay more attention to rare winner rows.
    pos = max(int(y_train.sum()), 1)
    neg = max(int((y_train == 0).sum()), 1)
    scale_pos_weight = neg / pos

    model = build_model(
        model_name=model_name,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        scale_pos_weight=scale_pos_weight,
    )
    model.fit(X_train, y_train)

    scored_df = test_df.copy()
    scored_df["pred_prob"] = model.predict_proba(X_test)[:, 1]
    metrics = score_predictions(scored_df)
    return model, scored_df, metrics


def rank_results(results_df: pd.DataFrame) -> pd.DataFrame:
    ranked_df = results_df.sort_values(
        by=[
            "game_level_accuracy",
            "top3_accuracy",
            "row_prauc",
            "row_recall",
            "row_precision",
            "row_auc",
            "row_brier",
            "row_logloss",
            "row_accuracy",
        ],
        ascending=[False, False, False, False, False, False, True, True, False],
    ).reset_index(drop=True)
    ranked_df.insert(0, "rank", range(1, len(ranked_df) + 1))
    return ranked_df

In [27]:
# Load the data, inspect the shared feature set, and display the metric hierarchy.
df = load_dataset()
numeric_cols, categorical_cols = get_feature_columns(df)
model_names, unavailable_models = available_model_names()
train_df, latest_test_df = latest_season_holdout_split(df)
latest_season = sorted_seasons(df["season"].dropna().unique().tolist())[-1]

summary_df = pd.DataFrame(
    {
        "item": [
            "rows",
            "numeric_features",
            "categorical_features",
            "train_games",
            "test_games",
            "train_positive_rate",
            "test_season",
            "available_models",
        ],
        "value": [
            len(df),
            len(numeric_cols),
            len(categorical_cols),
            train_df["GAME_ID"].nunique(),
            latest_test_df["GAME_ID"].nunique(),
            round(train_df["y"].mean(), 4),
            latest_season,
            ", ".join(model_names),
        ],
    }
)

metric_hierarchy_df = pd.DataFrame(
    [
        {"tier": "primary", "metric": "game_level_accuracy", "direction": "higher", "why": "Did the top-ranked player actually win the award?"},
        {"tier": "primary", "metric": "top3_accuracy", "direction": "higher", "why": "Was the real winner in the model's top three players?"},
        {"tier": "secondary", "metric": "row_prauc", "direction": "higher", "why": "More useful than ROC AUC when winners are rare."},
        {"tier": "secondary", "metric": "row_recall", "direction": "higher", "why": "How often the model catches true winner rows."},
        {"tier": "secondary", "metric": "row_precision", "direction": "higher", "why": "How often predicted winner rows are truly winners."},
        {"tier": "probability", "metric": "row_brier", "direction": "lower", "why": "Checks whether predicted probabilities are numerically close to reality."},
        {"tier": "probability", "metric": "row_logloss", "direction": "lower", "why": "Punishes confident mistakes more strongly than Brier score."},
        {"tier": "probability", "metric": "row_auc", "direction": "higher", "why": "Measures overall ranking quality across thresholds."},
        {"tier": "reference", "metric": "row_accuracy", "direction": "higher", "why": "Useful context, but can look inflated in imbalanced data."},
    ]
)

display(summary_df)
display(metric_hierarchy_df)

if unavailable_models:
    print("Some optional models are unavailable in this environment:")
    for model_name, message in unavailable_models.items():
        print(f"- {model_name}: {message}")

,item,value
0,rows,8377
1,numeric_features,33
2,categorical_features,0
3,train_games,707
4,test_games,71
5,train_positive_rate,0.0929
6,test_season,2025-26
7,available_models,"logistic, ridge_logistic, lasso_logistic, rand..."


,tier,metric,direction,why
0,primary,game_level_accuracy,higher,Did the top-ranked player actually win the award?
1,primary,top3_accuracy,higher,Was the real winner in the model's top three p...
2,secondary,row_prauc,higher,More useful than ROC AUC when winners are rare.
3,secondary,row_recall,higher,How often the model catches true winner rows.
4,secondary,row_precision,higher,How often predicted winner rows are truly winn...
5,probability,row_brier,lower,Checks whether predicted probabilities are num...
6,probability,row_logloss,lower,Punishes confident mistakes more strongly than...
7,probability,row_auc,higher,Measures overall ranking quality across thresh...
8,reference,row_accuracy,higher,"Useful context, but can look inflated in imbal..."


In [28]:
# Walk-forward evaluation tests each future season using only earlier seasons for training.
# This is the most realistic multi-season comparison for these baseline models.
walk_forward_rows = []

for train_seasons, test_season in walk_forward_season_splits(df):
    split_train_df = df[df["season"].isin(train_seasons)].copy()
    split_test_df = df[df["season"] == test_season].copy()

    for model_name in model_names:
        _, _, metrics = fit_and_score_model(
            model_name,
            split_train_df,
            split_test_df,
            numeric_cols,
            categorical_cols,
        )
        walk_forward_rows.append(
            {
                "model": model_name,
                "test_season": test_season,
                **metrics,
            }
        )

walk_forward_df = pd.DataFrame(walk_forward_rows)
walk_forward_summary_df = (
    walk_forward_df.groupby("model", as_index=False)[
        [
            "game_level_accuracy",
            "top3_accuracy",
            "row_prauc",
            "row_recall",
            "row_precision",
            "row_brier",
            "row_logloss",
            "row_auc",
            "row_accuracy",
        ]
    ]
    .mean()
)
walk_forward_ranked_df = rank_results(walk_forward_summary_df)

display(walk_forward_df.round(4))
display(walk_forward_ranked_df.round(4))

/Users/teddytaussig/anaconda3/envs/COM328/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/teddytaussig/anaconda3/envs/COM328/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/teddytaussig/anaconda3/envs/COM328/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/teddytaussig/anaconda3/envs/COM328/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/teddytaussig/anaconda3/envs/COM328/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning

,model,test_season,game_level_accuracy,top3_accuracy,row_prauc,row_recall,row_precision,row_brier,row_logloss,row_auc,row_accuracy
0,logistic,2019-20,0.4861,0.8056,0.4654,0.7917,0.2822,0.1337,0.4201,0.8856,0.8022
1,ridge_logistic,2019-20,0.4861,0.7917,0.4659,0.7778,0.2772,0.1340,0.4200,0.8855,0.7998
2,lasso_logistic,2019-20,0.5000,0.7917,0.4662,0.7639,0.2709,0.1340,0.4198,0.8863,0.7960
3,random_forest,2019-20,0.4444,0.7917,0.4152,0.1528,0.6111,0.0641,0.2121,0.8762,0.9159
4,svm,2019-20,0.4306,0.7500,0.3771,0.1944,0.5600,0.0664,0.2241,0.8621,0.9147
5,lightgbm,2019-20,0.5139,0.7917,0.4305,0.4306,0.4493,0.0749,0.2965,0.8771,0.9023
6,xgboost,2019-20,0.5139,0.7639,0.4328,0.6944,0.3226,0.1060,0.3207,0.8787,0.8430
7,logistic,2020-21,0.2083,0.7083,0.2275,0.7361,0.2524,0.1633,0.5134,0.8116,0.7792
8,ridge_logistic,2020-21,0.2083,0.7083,0.2267,0.7361,0.2524,0.1634,0.5129,0.8113,0.7792
9,lasso_logistic,2020-21,0.2083,0.7083,0.2268,0.7361,0.2512,0.1632,0.5121,0.8113,0.7779


,rank,model,game_level_accuracy,top3_accuracy,row_prauc,row_recall,row_precision,row_brier,row_logloss,row_auc,row_accuracy
0,1,xgboost,0.3547,0.7344,0.3328,0.6913,0.2554,0.1374,0.3986,0.8348,0.7802
1,2,lightgbm,0.3361,0.7129,0.3068,0.3617,0.3138,0.0970,0.3296,0.8247,0.8643
2,3,lasso_logistic,0.3251,0.7369,0.3238,0.7893,0.2396,0.1717,0.5188,0.8365,0.7444
3,4,logistic,0.3249,0.7372,0.3237,0.7951,0.2417,0.1719,0.5195,0.8364,0.7456
4,5,ridge_logistic,0.3231,0.7352,0.3238,0.7931,0.2410,0.1718,0.5188,0.8363,0.7452
5,6,svm,0.3203,0.7191,0.2935,0.0802,0.3325,0.0752,0.2507,0.8266,0.9021
6,7,random_forest,0.2823,0.7065,0.2777,0.0314,0.2911,0.0756,0.2543,0.8218,0.9047


In [22]:
# Train each baseline model on all earlier seasons and evaluate on the newest season only.
# This gives a final holdout table that you can report alongside the walk-forward results.
latest_rows = []
latest_models = {}
latest_scored_outputs = {}

for model_name in model_names:
    model, scored_df, metrics = fit_and_score_model(
        model_name,
        train_df,
        latest_test_df,
        numeric_cols,
        categorical_cols,
    )
    latest_models[model_name] = model
    latest_scored_outputs[model_name] = scored_df
    latest_rows.append({"model": model_name, "test_season": latest_season, **metrics})

latest_results_df = pd.DataFrame(latest_rows)
latest_ranked_df = rank_results(latest_results_df)

display(latest_ranked_df.round(4))

,rank,model,test_season,game_level_accuracy,top3_accuracy,row_prauc,row_recall,row_precision,row_brier,row_logloss,row_auc,row_accuracy
0,1,lasso_logistic,2025-26,0.3521,0.7606,0.3703,0.8169,0.2407,0.1677,0.4950,0.8560,0.7451
1,2,ridge_logistic,2025-26,0.3521,0.7606,0.3699,0.8169,0.2397,0.1678,0.4954,0.8557,0.7438
2,3,logistic,2025-26,0.3521,0.7606,0.3694,0.8169,0.2407,0.1679,0.4956,0.8555,0.7451
3,4,svm,2025-26,0.3239,0.7324,0.3223,0.0704,0.3846,0.0720,0.2382,0.8487,0.9038
4,5,random_forest,2025-26,0.3099,0.7183,0.2940,0.0000,0.0000,0.0725,0.2392,0.8391,0.9064
5,6,xgboost,2025-26,0.3099,0.6761,0.3161,0.7465,0.2398,0.1472,0.4220,0.8315,0.7581


In [23]:
# Show the exact shared feature set used by all baseline models.
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

print("Numeric features")
display(pd.DataFrame({"numeric_feature": numeric_cols}))

print("Categorical features")
display(pd.DataFrame({"categorical_feature": categorical_cols}))

Numeric features


,numeric_feature
0,minutes_decimal
1,points
2,reboundsOffensive
3,reboundsDefensive
4,reboundsTotal
5,assists
6,steals
7,blocks
8,turnovers
9,foulsPersonal


Categorical features


,categorical_feature


## Optuna Tuning

This section uses a `train / validation / test` setup:
- training seasons: fit candidate models
- validation season: choose hyperparameters with Optuna
- final test season: report tuned performance on unseen data

The Optuna objective uses the same metric hierarchy as the rest of the notebook, converted into one weighted score so the study can optimize it.

In [13]:
# Import Optuna if it is available, then define the tuning split,
# tunable search spaces, and helper functions.
if "importlib" not in globals():
    import importlib
if "pd" not in globals():
    import pandas as pd
if "INPUT_PATH" not in globals():
    INPUT_PATH = "Tommy_Award_Player_Game_Table.csv"
if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 42
if "PRED_THRESHOLD" not in globals():
    PRED_THRESHOLD = 0.5

if "LogisticRegression" not in globals():
    from sklearn.linear_model import LogisticRegression
if "RandomForestClassifier" not in globals():
    from sklearn.ensemble import RandomForestClassifier
if "SVC" not in globals():
    from sklearn.svm import SVC
if "Pipeline" not in globals():
    from sklearn.pipeline import Pipeline
if "ColumnTransformer" not in globals():
    from sklearn.compose import ColumnTransformer
if "SimpleImputer" not in globals():
    from sklearn.impute import SimpleImputer
if "StandardScaler" not in globals():
    from sklearn.preprocessing import StandardScaler
if "average_precision_score" not in globals():
    from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, precision_score, recall_score, roc_auc_score

if "build_preprocessor" not in globals():
    def build_preprocessor(numeric_cols, categorical_cols, scale_numeric):
        numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
        if scale_numeric:
            numeric_steps.append(("scaler", StandardScaler()))
        numeric_transformer = Pipeline(steps=numeric_steps)
        return ColumnTransformer(transformers=[("num", numeric_transformer, numeric_cols)])

if "scaled_model_names" not in globals():
    def scaled_model_names():
        return {"logistic", "ridge_logistic", "lasso_logistic", "svm"}

if "score_predictions" not in globals():
    def score_predictions(scored_df, threshold=PRED_THRESHOLD):
        pred_winners = (
            scored_df.sort_values(["GAME_ID", "pred_prob"], ascending=[True, False])
            .groupby("GAME_ID")
            .head(1)
            .copy()
        )
        top3 = (
            scored_df.sort_values(["GAME_ID", "pred_prob"], ascending=[True, False])
            .groupby("GAME_ID")
            .head(3)
            .copy()
        )
        y_true = scored_df["y"]
        y_prob = scored_df["pred_prob"].clip(1e-6, 1 - 1e-6)
        y_pred = (y_prob >= threshold).astype(int)
        return {
            "game_level_accuracy": pred_winners["y"].mean(),
            "top3_accuracy": top3.groupby("GAME_ID")["y"].max().mean(),
            "row_prauc": average_precision_score(y_true, y_prob),
            "row_recall": recall_score(y_true, y_pred, zero_division=0),
            "row_precision": precision_score(y_true, y_pred, zero_division=0),
            "row_brier": brier_score_loss(y_true, y_prob),
            "row_logloss": log_loss(y_true, y_prob, labels=[0, 1]),
            "row_auc": roc_auc_score(y_true, y_prob),
            "row_accuracy": (y_pred == y_true).mean(),
        }

if "rank_results" not in globals():
    def rank_results(results_df):
        ranked_df = results_df.sort_values(
            by=[
                "game_level_accuracy",
                "top3_accuracy",
                "row_prauc",
                "row_recall",
                "row_precision",
                "row_auc",
                "row_brier",
                "row_logloss",
                "row_accuracy",
            ],
            ascending=[False, False, False, False, False, False, True, True, False],
        ).reset_index(drop=True)
        ranked_df.insert(0, "rank", range(1, len(ranked_df) + 1))
        return ranked_df

try:
    import optuna
except Exception as exc:
    optuna = None
    optuna_import_error = str(exc).strip()
else:
    optuna_import_error = None


def latest_train_valid_test_split(df):
    seasons = sorted_seasons(df["season"].dropna().unique().tolist())
    if len(seasons) < 3:
        raise ValueError("Need at least three seasons for a train/validation/test split.")

    train_df = df[df["season"].isin(seasons[:-2])].copy()
    valid_df = df[df["season"] == seasons[-2]].copy()
    test_df = df[df["season"] == seasons[-1]].copy()
    return train_df, valid_df, test_df


def build_model_with_params(
    model_name: str,
    numeric_cols: list[str],
    categorical_cols: list[str],
    scale_pos_weight: float,
    model_params: dict | None = None,
):
    prep = build_preprocessor(
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        scale_numeric=model_name in scaled_model_names(),
    )
    params = dict(model_params or {})

    if model_name == "logistic":
        default_params = {
            "penalty": None,
            "solver": "lbfgs",
            "max_iter": 5000,
            "class_weight": "balanced",
            "random_state": RANDOM_STATE,
        }
        default_params.update(params)
        clf = LogisticRegression(**default_params)
    elif model_name == "ridge_logistic":
        default_params = {
            "penalty": "l2",
            "C": 1.0,
            "solver": "lbfgs",
            "max_iter": 5000,
            "class_weight": "balanced",
            "random_state": RANDOM_STATE,
        }
        default_params.update(params)
        clf = LogisticRegression(**default_params)
    elif model_name == "lasso_logistic":
        default_params = {
            "penalty": "l1",
            "C": 1.0,
            "solver": "liblinear",
            "max_iter": 5000,
            "class_weight": "balanced",
            "random_state": RANDOM_STATE,
        }
        default_params.update(params)
        clf = LogisticRegression(**default_params)
    elif model_name == "random_forest":
        default_params = {
            "n_estimators": 300,
            "class_weight": "balanced_subsample",
            "random_state": RANDOM_STATE,
            "n_jobs": -1,
        }
        default_params.update(params)
        clf = RandomForestClassifier(**default_params)
    elif model_name == "svm":
        default_params = {
            "kernel": "rbf",
            "probability": True,
            "class_weight": "balanced",
            "random_state": RANDOM_STATE,
        }
        default_params.update(params)
        clf = SVC(**default_params)
    elif model_name == "lightgbm":
        LGBMClassifier = importlib.import_module("lightgbm").LGBMClassifier
        default_params = {
            "n_estimators": 300,
            "learning_rate": 0.05,
            "class_weight": "balanced",
            "random_state": RANDOM_STATE,
            "verbose": -1,
        }
        default_params.update(params)
        clf = LGBMClassifier(**default_params)
    elif model_name == "xgboost":
        XGBClassifier = importlib.import_module("xgboost").XGBClassifier
        default_params = {
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "n_estimators": 300,
            "learning_rate": 0.05,
            "max_depth": 3,
            "subsample": 0.80,
            "colsample_bytree": 0.80,
            "random_state": RANDOM_STATE,
            "n_jobs": 4,
            "scale_pos_weight": scale_pos_weight,
        }
        default_params.update(params)
        clf = XGBClassifier(**default_params)
    else:
        raise ValueError(f"Unsupported model_name: {model_name}")

    return Pipeline(
        steps=[
            ("prep", prep),
            ("clf", clf),
        ]
    )


def fit_and_score_model_with_params(
    model_name: str,
    train_df,
    test_df,
    numeric_cols: list[str],
    categorical_cols: list[str],
    model_params: dict | None = None,
):
    feature_cols = numeric_cols + categorical_cols
    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]
    y_train = train_df["y"]

    pos = max(int(y_train.sum()), 1)
    neg = max(int((y_train == 0).sum()), 1)
    scale_pos_weight = neg / pos

    model = build_model_with_params(
        model_name=model_name,
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        scale_pos_weight=scale_pos_weight,
        model_params=model_params,
    )
    model.fit(X_train, y_train)

    scored_df = test_df.copy()
    scored_df["pred_prob"] = model.predict_proba(X_test)[:, 1]
    metrics = score_predictions(scored_df)
    return model, scored_df, metrics


def tunable_model_names(model_names: list[str]) -> list[str]:
    return [model_name for model_name in model_names if model_name != "logistic"]


def tuning_objective_score(metrics: dict[str, float]) -> float:
    return (
        1000.0 * float(metrics["game_level_accuracy"])
        + 100.0 * float(metrics["top3_accuracy"])
        + 10.0 * float(metrics["row_prauc"])
        + 5.0 * float(metrics["row_recall"])
        + 3.0 * float(metrics["row_precision"])
        + 1.0 * float(metrics["row_auc"])
        - 1.0 * float(metrics["row_brier"])
        - 1.0 * float(metrics["row_logloss"])
    )


def suggest_model_params(
    trial,
    model_name: str,
    scale_pos_weight: float,
) -> dict[str, object]:
    if model_name == "ridge_logistic":
        return {
            "C": trial.suggest_float("C", 1e-3, 100.0, log=True),
        }
    if model_name == "lasso_logistic":
        return {
            "C": trial.suggest_float("C", 1e-3, 20.0, log=True),
        }
    if model_name == "random_forest":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 150, 600, step=50),
            "max_depth": trial.suggest_int("max_depth", 3, 18),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            "class_weight": trial.suggest_categorical("class_weight", ["balanced", "balanced_subsample"]),
        }
    if model_name == "svm":
        return {
            "C": trial.suggest_float("C", 1e-3, 50.0, log=True),
            "gamma": trial.suggest_float("gamma", 1e-4, 1.0, log=True),
        }
    if model_name == "lightgbm":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 150, 600, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 15, 127),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        }
    if model_name == "xgboost":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 150, 600, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "gamma": trial.suggest_float("gamma", 0.0, 3.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "scale_pos_weight": scale_pos_weight,
        }
    raise ValueError(f"No Optuna search space defined for {model_name}")

In [14]:
# Configure the Optuna studies and inspect the train/validation/test split.
# This cell rebuilds the key notebook variables if you run it out of order.
if "importlib" not in globals():
    import importlib
if "pd" not in globals():
    import pandas as pd
if "display" not in globals():
    from IPython.display import display
if "INPUT_PATH" not in globals():
    INPUT_PATH = "Tommy_Award_Player_Game_Table.csv"
if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 42
if "MIN_TRAIN_SEASONS" not in globals():
    MIN_TRAIN_SEASONS = 3

if "optuna_import_error" not in globals():
    try:
        import optuna
    except Exception as exc:
        optuna = None
        optuna_import_error = str(exc).strip()
    else:
        optuna_import_error = None

if "sorted_seasons" not in globals():
    def sorted_seasons(seasons: list[str]) -> list[str]:
        return sorted(seasons, key=lambda season: int(str(season).split("-")[0]))

if "latest_train_valid_test_split" not in globals():
    def latest_train_valid_test_split(df):
        seasons = sorted_seasons(df["season"].dropna().unique().tolist())
        if len(seasons) < 3:
            raise ValueError("Need at least three seasons for a train/validation/test split.")

        train_df = df[df["season"].isin(seasons[:-2])].copy()
        valid_df = df[df["season"] == seasons[-2]].copy()
        test_df = df[df["season"] == seasons[-1]].copy()
        return train_df, valid_df, test_df

if "tunable_model_names" not in globals():
    def tunable_model_names(model_names: list[str]) -> list[str]:
        return [model_name for model_name in model_names if model_name != "logistic"]

if "load_dataset" not in globals():
    def load_dataset(path: str = INPUT_PATH):
        df = pd.read_csv(path, dtype={"GAME_ID": str}).copy()
        df["game_date"] = pd.to_datetime(df["game_date"], format="mixed")
        df = df[df["minutes_decimal"] > 0].copy()

        mins = df["minutes_decimal"].clip(lower=1e-6)
        if "hustle_proxy" not in df.columns:
            df["hustle_proxy"] = df["reboundsOffensive"] + df["steals"] + df["blocks"]
        if "stocks_per_min" not in df.columns:
            df["stocks_per_min"] = df["stocks"] / mins
        if "stocks_rank" not in df.columns:
            df["stocks_rank"] = df.groupby("GAME_ID")["stocks"].rank(method="min", ascending=False)
        if "hustle_proxy_rank" not in df.columns:
            df["hustle_proxy_rank"] = df.groupby("GAME_ID")["hustle_proxy"].rank(method="min", ascending=False)

        if "season" not in df.columns:
            start_year = df["game_date"].dt.year.where(df["game_date"].dt.month >= 10, df["game_date"].dt.year - 1)
            df["season"] = start_year.astype(str) + "-" + (start_year + 1).astype(str).str[-2:]

        return df

if "get_feature_columns" not in globals():
    def get_feature_columns(df):
        numeric_cols = [
            "minutes_decimal",
            "points",
            "reboundsOffensive",
            "reboundsDefensive",
            "reboundsTotal",
            "assists",
            "steals",
            "blocks",
            "turnovers",
            "foulsPersonal",
            "plusMinusPoints",
            "fieldGoalsMade",
            "fieldGoalsAttempted",
            "threePointersMade",
            "threePointersAttempted",
            "freeThrowsMade",
            "stocks",
            "points_per_min",
            "oreb_per_min",
            "reb_per_min",
            "ast_per_min",
            "stocks_per_min",
            "hustle_proxy",
            "points_rank",
            "reboundsOffensive_rank",
            "reboundsTotal_rank",
            "assists_rank",
            "steals_rank",
            "blocks_rank",
            "plusMinusPoints_rank",
            "minutes_decimal_rank",
            "stocks_rank",
            "hustle_proxy_rank",
        ]
        return numeric_cols, []

if "available_model_names" not in globals():
    def available_model_names():
        model_names = [
            "logistic",
            "ridge_logistic",
            "lasso_logistic",
            "random_forest",
            "svm",
        ]
        unavailable = {}

        for optional_model, module_name in [("lightgbm", "lightgbm"), ("xgboost", "xgboost")]:
            try:
                importlib.import_module(module_name)
                model_names.append(optional_model)
            except Exception as exc:
                unavailable[optional_model] = str(exc).strip()

        return model_names, unavailable

if "df" not in globals():
    df = load_dataset()
if "numeric_cols" not in globals() or "categorical_cols" not in globals():
    numeric_cols, categorical_cols = get_feature_columns(df)
if "model_names" not in globals() or "unavailable_models" not in globals():
    model_names, unavailable_models = available_model_names()

OPTUNA_TRIALS = 25
OPTUNA_TIMEOUT = None
MODELS_TO_TUNE = tunable_model_names(model_names)

tune_train_df, tune_valid_df, tune_test_df = latest_train_valid_test_split(df)
tune_seasons = sorted_seasons(df["season"].dropna().unique().tolist())

optuna_setup_df = pd.DataFrame(
    {
        "item": [
            "train_seasons",
            "validation_season",
            "test_season",
            "optuna_trials_per_model",
            "models_to_tune",
            "optuna_available",
        ],
        "value": [
            ", ".join(tune_seasons[:-2]),
            tune_seasons[-2],
            tune_seasons[-1],
            OPTUNA_TRIALS,
            ", ".join(MODELS_TO_TUNE),
            optuna_import_error is None,
        ],
    }
)

display(optuna_setup_df)

if unavailable_models:
    print("Unavailable optional models in this environment:")
    for model_name, message in unavailable_models.items():
        print(f"- {model_name}: {message}")

if optuna_import_error is not None:
    print("Optuna is not installed in this notebook environment.")
    print(optuna_import_error)
    print("Install it in the same Python environment as this notebook, then rerun this section.")

,item,value
0,train_seasons,"2016-17, 2017-18, 2018-19, 2019-20, 2020-21, 2..."
1,validation_season,2024-25
2,test_season,2025-26
3,optuna_trials_per_model,25
4,models_to_tune,"ridge_logistic, lasso_logistic, random_forest,..."
5,optuna_available,True


Unavailable optional models in this environment:
- lightgbm: No module named 'lightgbm'


In [15]:
# Run one Optuna study per model on the validation season.
# The best trial for each model is stored and summarized below.
if "optuna_import_error" not in globals():
    try:
        import optuna
    except Exception as exc:
        optuna = None
        optuna_import_error = str(exc).strip()
    else:
        optuna_import_error = None

if "df" not in globals():
    df = load_dataset()
if "numeric_cols" not in globals() or "categorical_cols" not in globals():
    numeric_cols, categorical_cols = get_feature_columns(df)
if "model_names" not in globals() or "unavailable_models" not in globals():
    model_names, unavailable_models = available_model_names()
if "MODELS_TO_TUNE" not in globals():
    MODELS_TO_TUNE = tunable_model_names(model_names)
if "tune_train_df" not in globals() or "tune_valid_df" not in globals() or "tune_test_df" not in globals():
    tune_train_df, tune_valid_df, tune_test_df = latest_train_valid_test_split(df)
if "OPTUNA_TRIALS" not in globals():
    OPTUNA_TRIALS = 25
if "OPTUNA_TIMEOUT" not in globals():
    OPTUNA_TIMEOUT = None

required_names = [
    "suggest_model_params",
    "fit_and_score_model_with_params",
    "tuning_objective_score",
    "rank_results",
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError(
        "Run the Optuna helper cell above first. Missing: "
        + ", ".join(missing_names)
    )

if optuna_import_error is None:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    tuned_studies = {}
    best_params_by_model = {}
    tuned_validation_rows = []

    tune_pos = max(int(tune_train_df["y"].sum()), 1)
    tune_neg = max(int((tune_train_df["y"] == 0).sum()), 1)
    tune_scale_pos_weight = tune_neg / tune_pos

    for model_name in MODELS_TO_TUNE:
        def objective(trial, current_model_name=model_name):
            trial_params = suggest_model_params(trial, current_model_name, tune_scale_pos_weight)
            _, _, metrics = fit_and_score_model_with_params(
                current_model_name,
                tune_train_df,
                tune_valid_df,
                numeric_cols,
                categorical_cols,
                model_params=trial_params,
            )
            trial.set_user_attr(
                "metrics",
                {key: float(value) for key, value in metrics.items()},
            )
            return tuning_objective_score(metrics)

        study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
            study_name=f"baseline_{model_name}",
        )
        study.optimize(objective, n_trials=OPTUNA_TRIALS, timeout=OPTUNA_TIMEOUT, show_progress_bar=False)

        tuned_studies[model_name] = study
        best_params_by_model[model_name] = study.best_trial.params
        tuned_validation_rows.append(
            {
                "model": model_name,
                "optuna_score": study.best_value,
                **study.best_trial.user_attrs["metrics"],
                "best_params": str(study.best_trial.params),
            }
        )

    tuned_validation_df = pd.DataFrame(tuned_validation_rows)
    tuned_validation_ranked_df = rank_results(
        tuned_validation_df.drop(columns=["optuna_score", "best_params"])
    ).merge(
        tuned_validation_df[["model", "optuna_score", "best_params"]],
        on="model",
        how="left",
    )
    display(tuned_validation_ranked_df.round(4))

,rank,model,game_level_accuracy,top3_accuracy,row_prauc,row_recall,row_precision,row_brier,row_logloss,row_auc,row_accuracy,optuna_score,best_params
0,1,xgboost,0.4268,0.7683,0.4035,0.8171,0.2248,0.1667,0.4665,0.8495,0.7153,512.6697,"{'n_estimators': 250, 'learning_rate': 0.08159..."
1,2,svm,0.4024,0.7439,0.3708,0.1463,0.5714,0.0719,0.2346,0.8593,0.9086,483.5362,"{'C': 45.212309123133075, 'gamma': 0.003684279..."
2,3,random_forest,0.3659,0.7073,0.3130,0.5122,0.2745,0.1076,0.3253,0.8311,0.8252,443.4979,"{'n_estimators': 150, 'max_depth': 16, 'min_sa..."
3,4,lasso_logistic,0.3415,0.7561,0.3395,0.8537,0.2349,0.1852,0.5482,0.8392,0.7222,425.5468,{'C': 0.8952940355412096}
4,5,ridge_logistic,0.3415,0.7561,0.3379,0.8537,0.2333,0.1848,0.5450,0.8388,0.7199,425.5296,{'C': 0.2682515641207125}


In [29]:
# Refit each tuned model on train+validation seasons, then evaluate once on the untouched test season.
# Plain logistic is kept as an untuned baseline reference.
if "optuna_import_error" not in globals():
    try:
        import optuna
    except Exception as exc:
        optuna = None
        optuna_import_error = str(exc).strip()
    else:
        optuna_import_error = None

if "df" not in globals():
    df = load_dataset()
if "numeric_cols" not in globals() or "categorical_cols" not in globals():
    numeric_cols, categorical_cols = get_feature_columns(df)
if "model_names" not in globals() or "unavailable_models" not in globals():
    model_names, unavailable_models = available_model_names()
if "tune_train_df" not in globals() or "tune_valid_df" not in globals() or "tune_test_df" not in globals():
    tune_train_df, tune_valid_df, tune_test_df = latest_train_valid_test_split(df)
if "tune_seasons" not in globals():
    tune_seasons = sorted_seasons(df["season"].dropna().unique().tolist())

required_names = [
    "best_params_by_model",
    "fit_and_score_model_with_params",
    "rank_results",
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError(
        "Run the Optuna study cell above first. Missing: "
        + ", ".join(missing_names)
    )

if optuna_import_error is None:
    final_train_df = pd.concat([tune_train_df, tune_valid_df], ignore_index=True)
    tuned_test_rows = []
    tuned_test_models = {}

    for model_name in model_names:
        trial_params = best_params_by_model.get(model_name)
        model, scored_df, metrics = fit_and_score_model_with_params(
            model_name,
            final_train_df,
            tune_test_df,
            numeric_cols,
            categorical_cols,
            model_params=trial_params,
        )
        tuned_test_models[model_name] = model
        tuned_test_rows.append(
            {
                "model": model_name,
                "status": "tuned" if trial_params is not None else "baseline",
                "test_season": tune_seasons[-1],
                **metrics,
            }
        )

    tuned_test_df = pd.DataFrame(tuned_test_rows)
    tuned_test_ranked_df = rank_results(tuned_test_df)
    display(tuned_test_ranked_df.round(4))

/Users/teddytaussig/anaconda3/envs/COM328/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,rank,model,status,test_season,game_level_accuracy,top3_accuracy,row_prauc,row_recall,row_precision,row_brier,row_logloss,row_auc,row_accuracy
0,1,svm,tuned,2025-26,0.3944,0.6901,0.3586,0.0845,0.4286,0.0705,0.2412,0.8355,0.9051
1,2,random_forest,tuned,2025-26,0.3662,0.7183,0.3322,0.5352,0.2879,0.1042,0.3197,0.8409,0.8349
2,3,lasso_logistic,tuned,2025-26,0.3521,0.7606,0.3706,0.8169,0.2407,0.1677,0.4950,0.8560,0.7451
3,4,logistic,baseline,2025-26,0.3521,0.7606,0.3694,0.8169,0.2407,0.1679,0.4956,0.8555,0.7451
4,5,ridge_logistic,tuned,2025-26,0.3521,0.7324,0.3689,0.8169,0.2397,0.1676,0.4947,0.8566,0.7438
5,6,xgboost,tuned,2025-26,0.3380,0.6901,0.3281,0.8028,0.2308,0.1621,0.4587,0.8365,0.7347
6,7,lightgbm,baseline,2025-26,0.3380,0.6901,0.3184,0.4648,0.3028,0.0982,0.3061,0.8388,0.8518
